# HQ-EEGNet
### Hybrid Quantum–Classical Deep Learning for Alzheimer's Disease Detection from Resting-State EEG

**Author:** Prachi · MSc Computer Science, University of Delhi

This notebook contains the HQ-EEGNet implementation: time–frequency feature extraction (spectrogram, scalogram, Hilbert features), a subject-level dataset with variable-length epoch padding, three CNN encoders with attention pooling, a 16-qubit variational quantum circuit (PennyLane), and stratified 5-fold subject-level cross-validation.

See `README.md` for a full description of the method.

*© 2026 Prachi. All rights reserved. Shared for review only.*

## 1. Imports

In [ ]:
import os
import numpy as np
import cv2
import pywt
import tqdm
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
from scipy.signal import spectrogram, get_window, hilbert, butter, filtfilt, iirnotch
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, accuracy_score, recall_score
from joblib import load, dump
import pennylane as qml

## 2. Configuration

In [ ]:
TARGET_SFREQ  = 64      # Hz, sampling rate after downsampling
IMG_SIZE      = 32      # time–frequency image size (32 x 32)
FREQ_LOW      = 1.0     # Hz
FREQ_HIGH     = 30.0    # Hz
NOTCH_FREQ    = 50.0    # Hz, power-line frequency
NOTCH_Q       = 30.0

N_QUBITS      = 16
FEAT_DIM      = 128     # embedding size per CNN branch
BATCH_SIZE    = 4       # subjects per batch
NUM_EPOCHS    = 300
WARMUP_EPOCHS = 10      # quantum layer frozen for these epochs
N_FOLDS       = 5
SEED          = 42

torch.manual_seed(SEED)
np.random.seed(SEED)

## 3. Signal preprocessing

In [ ]:
def norm01(x, eps=1e-8):
    x = x - x.min()
    return x / (x.max() + eps)


def resize(x):
    return cv2.resize(x.astype(np.float32), (IMG_SIZE, IMG_SIZE))


def bandpass_filter(signal, sfreq=TARGET_SFREQ, low=FREQ_LOW, high=FREQ_HIGH, order=4):
    nyq = sfreq / 2.0
    b, a = butter(order, [low / nyq, high / nyq], btype="band")
    return filtfilt(b, a, signal)


def notch_filter(signal, sfreq=TARGET_SFREQ, freq=NOTCH_FREQ, Q=NOTCH_Q):
    if freq >= sfreq / 2.0:          # skip if above Nyquist
        return signal
    b, a = iirnotch(freq / (sfreq / 2.0), Q)
    return filtfilt(b, a, signal)


def preprocess(signal, sfreq=TARGET_SFREQ):
    signal = notch_filter(signal, sfreq)
    signal = bandpass_filter(signal, sfreq)
    return signal

## 4. Time–frequency representations

In [ ]:
def compute_spectrogram(signal, sfreq=TARGET_SFREQ):
    win = int(1.0 * sfreq)
    hop = int(0.5 * sfreq)
    f, _, Sxx = spectrogram(
        signal,
        fs=sfreq,
        window=get_window("hamming", win),
        nperseg=win,
        noverlap=win - hop,
        mode="psd",
    )
    mask = (f >= FREQ_LOW) & (f <= FREQ_HIGH)
    Sxx  = np.log1p(Sxx[mask])
    return resize(norm01(Sxx))


def compute_scalogram(signal, sfreq=TARGET_SFREQ, wavelet="morl", num_scales=IMG_SIZE):
    central_freq = pywt.central_frequency(wavelet)
    freq_axis    = np.linspace(FREQ_HIGH, FREQ_LOW, num_scales)
    scales       = central_freq * sfreq / freq_axis
    coeffs, _    = pywt.cwt(signal, scales, wavelet, sampling_period=1.0 / sfreq)
    S = np.log1p(np.abs(coeffs) ** 2)
    return resize(norm01(S))


def compute_hilbert_features(signal, sfreq=TARGET_SFREQ):
    analytic      = hilbert(signal)
    envelope      = np.abs(analytic)
    inst_phase    = np.unwrap(np.angle(analytic))
    inst_freq     = np.diff(inst_phase, prepend=inst_phase[0]) * sfreq / (2.0 * np.pi)
    env_n         = norm01(np.log1p(envelope))
    freq_n        = norm01(np.clip(inst_freq, FREQ_LOW, FREQ_HIGH))
    combined      = np.stack([env_n, freq_n], axis=0)   # (2, T)
    return resize(combined.T)                            # (T, 2) -> (H, W)


def build_feature_stack(epoch, sfreq=TARGET_SFREQ):
    spec, scal, hilb = [], [], []
    for c in range(epoch.shape[0]):
        sig = preprocess(epoch[c], sfreq)
        spec.append(compute_spectrogram(sig, sfreq))
        scal.append(compute_scalogram(sig, sfreq))
        hilb.append(compute_hilbert_features(sig, sfreq))
    return {
        "spectrogram": np.stack(spec),
        "scalogram":   np.stack(scal),
        "hilbert":     np.stack(hilb),
    }


def process_all(data_list):
    results = []
    for subject in tqdm.tqdm(data_list, desc="Extracting features"):
        subj_id = subject["participant_id"]
        epochs  = subject["epochs"].get_data()      # (E, C, T)

        spec_list, scal_list, hilb_list = [], [], []
        for epoch in epochs:
            feats = build_feature_stack(epoch)
            spec_list.append(feats["spectrogram"])
            scal_list.append(feats["scalogram"])
            hilb_list.append(feats["hilbert"])

        results.append({
            "participant_id": subj_id,
            "group_num":      subject["group_num"],
            "spectrogram":    np.array(spec_list, dtype=np.float32),
            "scalogram":      np.array(scal_list, dtype=np.float32),
            "hilbert":        np.array(hilb_list, dtype=np.float32),
        })
    return results

## 5. Subject-level dataset and variable-length batching

In [ ]:
class EEGDataset(Dataset):
    def __init__(self, data_list, train=True):
        self.data  = data_list
        self.train = train

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        sample = self.data[idx]
        x1 = torch.tensor(sample["spectrogram"], dtype=torch.float32)  # (E, C, H, W)
        x2 = torch.tensor(sample["scalogram"],   dtype=torch.float32)
        x3 = torch.tensor(sample["hilbert"],     dtype=torch.float32)

        if self.train:
            # Gaussian noise augmentation
            x1 = x1 + torch.randn_like(x1) * 0.01
            x2 = x2 + torch.randn_like(x2) * 0.01
            x3 = x3 + torch.randn_like(x3) * 0.01
            # Random channel dropout
            if torch.rand(1).item() < 0.2:
                c = torch.randint(0, x1.shape[1], (1,)).item()
                x1[:, c] = 0.0
                x2[:, c] = 0.0
                x3[:, c] = 0.0

        y = torch.tensor(sample["group_num"], dtype=torch.long)
        return x1, x2, x3, y


def collate_fn(batch):
    x1_list, x2_list, x3_list, ys = zip(*batch)

    lengths = torch.tensor([x.shape[0] for x in x1_list])

    # Pad along the epoch (frame) dimension
    x1 = pad_sequence([x.float() for x in x1_list], batch_first=True)
    x2 = pad_sequence([x.float() for x in x2_list], batch_first=True)
    x3 = pad_sequence([x.float() for x in x3_list], batch_first=True)

    F_max = x1.shape[1]
    mask  = torch.arange(F_max).unsqueeze(0) >= lengths.unsqueeze(1)  # (B, F_max)

    labels = torch.stack(ys)
    return x1, x2, x3, labels, mask, lengths

## 6. Quantum layer (16-qubit variational circuit)

In [ ]:
dev = qml.device("default.qubit", wires=N_QUBITS)

@qml.qnode(dev, interface="torch", diff_method="backprop")
def quantum_circuit(inputs, weights):
    # Angle encoding
    for i in range(N_QUBITS):
        qml.RY(inputs[i], wires=i)
        qml.RZ(inputs[i] * 0.5, wires=i)
    # Variational layers: general rotations + ring of CNOTs
    for layer in range(weights.shape[0]):
        for i in range(N_QUBITS):
            qml.Rot(*weights[layer, i], wires=i)
        for i in range(N_QUBITS):
            qml.CNOT(wires=[i, (i + 1) % N_QUBITS])
    return [qml.expval(qml.PauliZ(i)) for i in range(N_QUBITS)]


class HybridQNN(nn.Module):
    def __init__(self, n_layers=3):
        super().__init__()
        self.q_weights = nn.Parameter(torch.randn(n_layers, N_QUBITS, 3) * 0.01)
        self.proj      = nn.Linear(N_QUBITS, N_QUBITS)
        self.out_scale = nn.Parameter(torch.ones(N_QUBITS))
        self.out_bias  = nn.Parameter(torch.zeros(N_QUBITS))
        self.residual  = nn.Linear(N_QUBITS, N_QUBITS)

    def forward(self, x):
        B        = x.shape[0]
        residual = self.residual(x)
        x_proj   = torch.clamp(self.proj(x), -np.pi / 2, np.pi / 2)

        x_cpu = x_proj.cpu()
        w_cpu = self.q_weights.cpu()

        outputs = []
        for i in range(B):
            result = quantum_circuit(x_cpu[i], w_cpu)
            outputs.append(torch.stack(result))

        q_out = torch.stack(outputs).float().to(residual.device)
        q_out = q_out * self.out_scale + self.out_bias
        return q_out + residual

## 7. CNN encoders, attention pooling and the full hybrid model

In [ ]:
class Encoder(nn.Module):
    def __init__(self, in_channels=19, feat_dim=FEAT_DIM):
        super().__init__()
        self.extractor = nn.Sequential(
            nn.Conv2d(in_channels, 32, 3, padding=1),
            nn.BatchNorm2d(32), nn.GELU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, 3, padding=1),
            nn.BatchNorm2d(64), nn.GELU(),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 32, 3, padding=1),
            nn.BatchNorm2d(32), nn.GELU(),
            nn.AdaptiveAvgPool2d((1, 1)),
        )
        self.proj = nn.Sequential(
            nn.Linear(32, 64),
            nn.GELU(),
            nn.Dropout(0.3),
            nn.Linear(64, feat_dim),
        )

    def forward(self, x):
        x = self.extractor(x)
        x = x.view(x.size(0), -1)
        return self.proj(x)


class Model(nn.Module):
    def __init__(self, n_channels=19):
        super().__init__()
        self.spectrogram_cnn = Encoder(in_channels=n_channels)
        self.scalogram_cnn   = Encoder(in_channels=n_channels)
        self.hilbert_cnn     = Encoder(in_channels=n_channels)

        # Attention pooling over epoch frames
        self.attn = nn.Linear(FEAT_DIM, 1)

        self.classifier = nn.Sequential(
            nn.Linear(FEAT_DIM * 3, 256),
            nn.GELU(),
            nn.Dropout(0.4),
            nn.Linear(256, 64),
            nn.GELU(),
            nn.Linear(64, N_QUBITS),
        )
        self.qnn = HybridQNN(n_layers=3)
        self.out = nn.Linear(N_QUBITS, 1)

    def encode_branch(self, x, encoder, mask=None):
        B, F, C, H, W = x.shape
        feats = encoder(x.view(B * F, C, H, W))    # (B*F, feat_dim)
        feats = feats.view(B, F, -1)                # (B, F, feat_dim)

        # Attention pooling
        attn_w = self.attn(feats)                   # (B, F, 1)
        if mask is not None:
            attn_w = attn_w.masked_fill(mask.unsqueeze(-1), float("-inf"))
        attn_w = torch.softmax(attn_w, dim=1)
        return (feats * attn_w).sum(dim=1)          # (B, feat_dim)

    def forward(self, sp, sc, h, mask=None):
        sp = self.encode_branch(sp, self.spectrogram_cnn, mask)
        sc = self.encode_branch(sc, self.scalogram_cnn,   mask)
        h  = self.encode_branch(h,  self.hilbert_cnn,     mask)

        x = torch.cat([sp, sc, h], dim=1)          # (B, feat_dim*3)
        x = self.classifier(x)
        x = self.qnn(x)
        return self.out(x.float())                  # (B, 1)

## 8. Training

In [ ]:
def smooth_bce(logits, labels, eps=0.1):
    """Label-smoothed binary cross-entropy."""
    labels_smooth = labels * (1 - eps) + 0.5 * eps
    return F.binary_cross_entropy_with_logits(logits, labels_smooth)


def training(train_loader, test_loader, n_channels=19,
             num_epochs=NUM_EPOCHS, warmup_epochs=WARMUP_EPOCHS):

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")

    model = Model(n_channels=n_channels).to(device)

    all_labels = torch.cat([label for *_, label, _mask, _len in train_loader]).float()
    pos_count  = all_labels.sum().clamp(min=1)
    neg_count  = (len(all_labels) - pos_count).clamp(min=1)
    pos_weight = (neg_count / pos_count).unsqueeze(0).to(device)
    print(f"Dataset: {int(pos_count)} pos, {int(neg_count)} neg | pos_weight={pos_weight.item():.3f}")

    quantum_params   = list(model.qnn.parameters())
    classical_params = [p for n, p in model.named_parameters() if "qnn" not in n]

    optimizer = torch.optim.AdamW([
        {"params": classical_params,       "lr": 1e-3, "weight_decay": 1e-4},
        {"params": model.qnn.q_weights,    "lr": 0.05, "weight_decay": 0.0},
        {"params": [p for p in quantum_params if p is not model.qnn.q_weights],
                                           "lr": 1e-3, "weight_decay": 1e-4},
    ])

    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=num_epochs, eta_min=1e-5
    )

    best_f1    = 0.0
    best_state = None

    for epoch in tqdm.tqdm(range(num_epochs), desc="Epochs"):
        # Freeze / unfreeze quantum weights
        for p in model.qnn.parameters():
            p.requires_grad_(epoch >= warmup_epochs)

        model.train()
        epoch_loss = 0.0

        for x1, x2, x3, label, mask, lengths in train_loader:
            x1    = x1.to(device)
            x2    = x2.to(device)
            x3    = x3.to(device)
            label = label.float().to(device)
            mask  = mask.to(device)

            optimizer.zero_grad()
            logits = model(x1, x2, x3, mask=mask).squeeze(-1)

            # Mix label-smoothed loss + weighted BCE
            loss_smooth   = smooth_bce(logits, label, eps=0.1)
            loss_weighted = F.binary_cross_entropy_with_logits(
                logits, label,
                pos_weight=pos_weight.expand(label.shape[0])
            )
            loss = 0.5 * loss_smooth + 0.5 * loss_weighted

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            epoch_loss += loss.item()

        scheduler.step()

        model.eval()
        true_labels, all_probs = [], []

        with torch.no_grad():
            for x1, x2, x3, label, mask, lengths in test_loader:
                x1   = x1.to(device)
                x2   = x2.to(device)
                x3   = x3.to(device)
                mask = mask.to(device)
                probs = torch.sigmoid(model(x1, x2, x3, mask=mask).squeeze(-1))
                all_probs.extend(probs.cpu().tolist())
                true_labels.extend(label.tolist())

        best_thresh, best_f1_epoch = 0.5, 0.0
        for thresh in np.arange(0.1, 0.91, 0.05):
            preds = [1 if p > thresh else 0 for p in all_probs]
            f = f1_score(true_labels, preds, zero_division=0)
            if f > best_f1_epoch:
                best_f1_epoch, best_thresh = f, thresh

        preds = [1 if p > best_thresh else 0 for p in all_probs]
        acc   = accuracy_score(true_labels, preds)
        sens  = recall_score(true_labels, preds, zero_division=0)
        spec  = recall_score(true_labels, preds, pos_label=0, zero_division=0)
        phase = "warmup" if epoch < warmup_epochs else "train"

        print(
            f"[{phase}] Epoch {epoch+1:03d} | "
            f"Loss: {epoch_loss/len(train_loader):.4f} | "
            f"Acc: {acc:.3f} | Sens: {sens:.3f} | Spec: {spec:.3f} | "
            f"F1: {best_f1_epoch:.3f} | Thresh: {best_thresh:.2f}"
        )

        if best_f1_epoch > best_f1:
            best_f1    = best_f1_epoch
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

    if best_state is not None:
        model.load_state_dict(best_state)
        print(f"\nBest F1: {best_f1:.3f} — best checkpoint restored.")

    return model

## 9. Stratified 5-fold subject-level cross-validation

In [ ]:
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

DATA       = load("./data.pkl")
labels     = np.array([d["group_num"] for d in DATA])
n_channels = DATA[0]["spectrogram"].shape[1]

all_metrics = []

for fold_id, (train_idx, val_idx) in enumerate(skf.split(np.arange(len(DATA)), labels)):
    print(f"\n{'-'*50}\nFOLD {fold_id+1}/{N_FOLDS} | Train: {len(train_idx)} | Val: {len(val_idx)}\n{'-'*50}")

    train_data = [DATA[i] for i in train_idx]
    val_data   = [DATA[i] for i in val_idx]

    train_loader = DataLoader(
        EEGDataset(train_data, train=True),
        batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn,
        drop_last=(len(train_data) % BATCH_SIZE == 1),
    )
    val_loader = DataLoader(
        EEGDataset(val_data, train=False),
        batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn,
    )

    model = training(train_loader, val_loader, n_channels=n_channels)
    torch.save(model, f"model_fold{fold_id+1}.pt")

    # Final evaluation on the validation fold
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.eval()
    true_labels, all_probs = [], []
    with torch.no_grad():
        for x1, x2, x3, label, mask, lengths in val_loader:
            probs = torch.sigmoid(model(x1.to(device), x2.to(device), x3.to(device), mask=mask.to(device)).squeeze(-1))
            all_probs.extend(probs.cpu().tolist())
            true_labels.extend(label.tolist())

    best_thresh, best_f1 = 0.5, 0.0
    for thresh in np.arange(0.1, 0.91, 0.05):
        preds = [1 if p > thresh else 0 for p in all_probs]
        f = f1_score(true_labels, preds, zero_division=0)
        if f > best_f1:
            best_f1, best_thresh = f, thresh

    preds = [1 if p > best_thresh else 0 for p in all_probs]
    fold_metrics = {
        "fold":     fold_id + 1,
        "accuracy": accuracy_score(true_labels, preds),
        "f1":       best_f1,
        "sens":     recall_score(true_labels, preds, zero_division=0),
        "spec":     recall_score(true_labels, preds, pos_label=0, zero_division=0),
        "thresh":   best_thresh,
    }
    all_metrics.append(fold_metrics)
    print(f"Fold {fold_id+1} | Acc: {fold_metrics['accuracy']:.3f} | F1: {fold_metrics['f1']:.3f} | Sens: {fold_metrics['sens']:.3f} | Spec: {fold_metrics['spec']:.3f}")


print("CV SUMMARY")
for key in ["accuracy", "f1", "sens", "spec"]:
    vals = [m[key] for m in all_metrics]
    print(f"{key:>10}: {np.mean(vals):.3f} ± {np.std(vals):.3f}")